In [1]:
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

In [4]:
from pyspark.sql import SparkSession  
spark = SparkSession.builder.appName("SalesData").getOrCreate()
sales_raw_df = (
        spark.read.format("csv")
        .option("header", "true")
        .schema(file_schema)
        .load("C:\\Users\\abhis\\OneDrive\\Documents\\Pyspark Project\\sales_sample.csv")

)

sales_raw_df.show(10)

+---+--------+----------+----------+----------+--------+
| id|    name|       dop|     phone|    amount|discount|
+---+--------+----------+----------+----------+--------+
|100|Prashant|2020-06-15|9238614990|     12000|    18.5|
|101|   David| 2018-08-7|8908617610|     15000|     nil|
|102|  Simran|14-05-2019|      NULL|3000000000|      21|
+---+--------+----------+----------+----------+--------+



Describing the data to understand it.


In [6]:
sales_raw_df.describe().show()

+-------+-----+------+----------+--------------------+--------------------+------------------+
|summary|   id|  name|       dop|               phone|              amount|          discount|
+-------+-----+------+----------+--------------------+--------------------+------------------+
|  count|    3|     3|         3|                   2|                   3|                 3|
|   mean|101.0|  NULL|      NULL|         9.0736163E9|          1.000009E9|             19.75|
| stddev|  1.0|  NULL|      NULL|2.3334338517179397E8|1.7320430133408928E9|1.7677669529663689|
|    min|  100| David|14-05-2019|          8908617610|               12000|              18.5|
|    max|  102|Simran|2020-06-15|          9238614990|          3000000000|               nil|
+-------+-----+------+----------+--------------------+--------------------+------------------+



After reading. 

1. There are 3 records
2. Why less records in phone field - because there is one null field.
3. Stddev and mean does not make sense as there is no more data to the dataset. At least all it tells me its numericals.
4. Date format is mixed with dd-mm-yyyy and yyyy-mm-dd 
5. after reading phone can have schema as long
6. a single invoice cannot have this large big amount. Data has outliers which means bogus records. 
7. Discount has MAX as nil - it can mean no discount that means the schema is character. 
8. Name field should be as 'Customer Name'

The corrections to be made : - 

1. Convert id from integer to string and rename it as transaction_id.
2. Rename the name column to customer_name.
3. Convert the dop to date format and rename the column to date_of_purchase.
4. Rename the phone column to customer_phone
5. Convert the amount to a long value and filter out nulls and outlier values. 
6. Rename the column to purchase_amount
7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

Prepare and clean the dataframe using appropriate transformations

In [18]:
from pyspark.sql.functions import col, to_date
result_Sales_df = (
    sales_raw_df.selectExpr(
        "cast(id as string) as transaction_id",
        "name as customer_name",
        "nvl(try_cast(dop as date), to_date(dop, 'yyyy-MM-dd')) as date_of_purchase",
        "cast(phone as string) as customer_phone",
        "cast(amount as long) as purchase_amount",
        "nvl(try_cast(discount as double),0) as discount"
    )
)

result_Sales_df.show(10)    

+--------------+-------------+----------------+--------------+---------------+--------+
|transaction_id|customer_name|date_of_purchase|customer_phone|purchase_amount|discount|
+--------------+-------------+----------------+--------------+---------------+--------+
|           100|     Prashant|      2020-06-15|    9238614990|          12000|    18.5|
|           101|        David|      2018-08-07|    8908617610|          15000|     0.0|
|           102|       Simran|            NULL|          NULL|     3000000000|    21.0|
+--------------+-------------+----------------+--------------+---------------+--------+



In [19]:
result_Sales_df.filter("purchase_amount is not null and purchase_amount < 200000").show()



+--------------+-------------+----------------+--------------+---------------+--------+
|transaction_id|customer_name|date_of_purchase|customer_phone|purchase_amount|discount|
+--------------+-------------+----------------+--------------+---------------+--------+
|           100|     Prashant|      2020-06-15|    9238614990|          12000|    18.5|
|           101|        David|      2018-08-07|    8908617610|          15000|     0.0|
+--------------+-------------+----------------+--------------+---------------+--------+



Now verify the statistics again after the EDA transformations

In [21]:
result_Sales_df.describe().show()

+-------+--------------+-------------+--------------------+--------------------+------------------+
|summary|transaction_id|customer_name|      customer_phone|     purchase_amount|          discount|
+-------+--------------+-------------+--------------------+--------------------+------------------+
|  count|             3|            3|                   2|                   3|                 3|
|   mean|         101.0|         NULL|         9.0736163E9|          1.000009E9|13.166666666666666|
| stddev|           1.0|         NULL|2.3334338517179397E8|1.7320430133408928E9|11.470977871713176|
|    min|           100|        David|          8908617610|               12000|               0.0|
|    max|           102|       Simran|          9238614990|          3000000000|              21.0|
+-------+--------------+-------------+--------------------+--------------------+------------------+

